# 01 Webcam-Based PM2.5 Dataset Pipeline

This notebook implements the complete workflow for constructing a webcam-based PM2.5 dataset, including:

- Webcam image loading and ROI selection
- Image feature extraction
- PM2.5, ERA5, and ARPA meteorological data processing
- Multi-source dataset merging and cleaning

The final output is an analysis-ready dataset for subsequent air quality modeling and analysis.


## Setup

In [2]:
import importlib
import json
import sys

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

plt.rcParams["font.family"] = "Times New Roman"

## 1. Load Collected Webcam Images
Load all collected webcam images and initialize the project paths.

In [4]:
def find_project_root(start_path=None):
    """Find the nearest parent containing the project source and notebooks."""
    start = Path.cwd() if start_path is None else Path(start_path)
    start = start.resolve()

    for candidate in (start, *start.parents):
        if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root containing src/ and notebooks/."
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Detect environment
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    IMAGE_DIR = Path("/content/drive/MyDrive/webcam_images")
else:
    IMAGE_DIR = PROJECT_ROOT / "data" / "raw" / "images"

image_files = sorted(IMAGE_DIR.glob("*.jpg"))

print("Running in Colab:", IN_COLAB)
print("Project root:", PROJECT_ROOT)
print(
    "Image directory:",
    IMAGE_DIR.relative_to(PROJECT_ROOT)
    if not IN_COLAB
    else IMAGE_DIR,
)
print("Number of images:", len(image_files))


Running in Colab: False
Project root: /Users/qingxuan/Desktop/webcam-pm25-benchmark
Image directory: data/raw/images
Number of images: 3862


## 2. ROI Extraction
Interactively select the region of interest (ROI) used for feature extraction.

In [5]:
import src.vision.roi as roi_viewer

importlib.reload(roi_viewer)

roi_controls = roi_viewer.build_roi_viewer(IMAGE_DIR)

Image(value=b'')

In [20]:
selected_roi = {
    "top": roi_controls["top"].value,
    "left": roi_controls["left"].value,
    "height": roi_controls["height"].value,
    "width": roi_controls["width"].value
}

ROI_JSON = PROJECT_ROOT / "config" / "roi.json"
ROI_JSON.parent.mkdir(parents=True, exist_ok=True)

with open(ROI_JSON, "w") as f:
    json.dump(selected_roi, f, indent=4)

print("Selected ROI:")
print(f"top = {selected_roi['top']}")
print(f"left = {selected_roi['left']}")
print(f"height = {selected_roi['height']}")
print(f"width = {selected_roi['width']}")
print(f"mode = {roi_controls['mode'].value}")

print("\nROI saved to:")
print(ROI_JSON.relative_to(PROJECT_ROOT))

Selected ROI:
top = 160
left = 116
height = 380
width = 515
mode = RGB

ROI saved to:
config/roi.json


## 3. Image Feature Extraction
Extract RGB, saturation, contrast, and B/R ratio features from all webcam images.

In [13]:
import src.vision.handcrafted as image_features

importlib.reload(image_features)

IMAGE_FEATURES_CSV = PROJECT_ROOT / "data" / "interim" / "image_features.csv"

rows, skipped_files = image_features.extract_image_features(
    image_dir=PROJECT_ROOT / "data" / "raw" / "images",
    output_csv=IMAGE_FEATURES_CSV,
    roi=selected_roi
)

df = pd.read_csv(IMAGE_FEATURES_CSV)

df["image_path"] = df["image_path"].apply(
    lambda x: str(Path(x).relative_to(PROJECT_ROOT))
)

df.to_csv(IMAGE_FEATURES_CSV, index=False)

print(f"Image features extracted: {len(rows)} rows")
print("Saved CSV:", IMAGE_FEATURES_CSV.relative_to(PROJECT_ROOT))
print(f"Skipped files: {len(skipped_files)}")

df.head()

Image features extracted: 3862 rows
Saved CSV: data/interim/image_features.csv
Skipped files: 0


,datetime,R_roi,G_roi,B_roi,R_std,G_std,B_std,S_mean,V_mean,colorfulness,sky_brightness,contrast,gray_entropy,laplacian_variance,mean_gradient_magnitude,local_contrast,B_R_ratio,dark_pixel_ratio,sky_luminance_gradient,image_path
0,2025-01-29 15:00:00,72.890291,102.643112,127.905830,33.593960,41.347283,45.609732,0.445059,0.501627,36.742349,0.612625,38.979534,7.058169,1438.938624,4.961617,10.883452,1.754772,0.198304,24.746829,data/raw/images/20250129-1600.jpg
1,2025-01-29 16:00:00,77.456367,108.505774,131.582800,37.263657,48.407776,57.147530,0.409142,0.517590,48.521689,0.692133,44.985222,7.000933,1500.896531,5.284602,11.230147,1.698799,0.236648,17.017330,data/raw/images/20250129-1700.jpg
2,2025-01-29 17:00:00,61.865319,81.209939,99.889218,36.117171,39.679788,46.476770,0.439677,0.409517,45.040358,0.505191,37.646871,7.040461,4203.778717,11.748291,21.471251,1.614624,0.210199,34.354465,data/raw/images/20250129-1800.jpg
3,2025-01-29 18:00:00,76.285973,96.945933,117.473485,40.910550,39.854581,42.496365,0.384557,0.467538,33.898834,0.460016,39.745321,6.890734,3863.428672,11.354737,21.494355,1.539909,0.076776,28.610417,data/raw/images/20250129-1900.jpg
4,2025-01-29 19:00:00,82.091906,102.342810,122.670874,35.199885,35.962358,41.029133,0.349021,0.485637,31.291664,0.501104,35.687420,6.790390,3685.742479,11.176274,19.922616,1.494311,0.075064,13.407924,data/raw/images/20250129-2000.jpg


## 4. Download and inspect PM2.5 data
Download hourly PM2.5 observations from the EEA API, then inspect their raw temporal coverage and distribution.

In [5]:
import src.data.pm25 as pm25

importlib.reload(pm25)

PM25_CSV = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "pm25"
    / "PM25_MI_hourly.csv"
)

PM25_TEMP_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "pm25_temp"
)

print("Downloading PM2.5 data for the requested date range.")

pm25_df = pm25.download_pm25_data(
    api_start="2025-01-01T00:00:00Z",
    api_end="2026-08-15T23:00:00Z",
    station_prefix="IT/SPO.IT0477A_6001_BETA",
    temp_dir=PM25_TEMP_DIR,
    output_file=PM25_CSV,
    remove_temp=True,
)

print("Saved CSV:", PM25_CSV.relative_to(PROJECT_ROOT))

print(f"Rows: {len(pm25_df)}")
pm25_df.head()

Saved CSV: data/raw/pm25/PM25_MI_hourly.csv
Rows: 14160


,Samplingpoint,Pollutant,Start,End,Value,Unit,AggType,Validity,Verification,ResultTime,DataCapture,FkObservationLog
0,IT/SPO.IT0477A_6001_BETA_2022-01-01_00:00:00,6001,2025-01-01 00:00:00,2025-01-01 01:00:00,630.703370000000000000,ug.m-3,hour,3,3,2025-01-02 18:00:00,None,ad50fa1d-3abd-4bb0-8360-a37d75654a61
1,IT/SPO.IT0477A_6001_BETA_2022-01-01_00:00:00,6001,2025-01-01 01:00:00,2025-01-01 02:00:00,519.935800000000000000,ug.m-3,hour,3,3,2025-01-02 18:00:00,None,ad50fa1d-3abd-4bb0-8360-a37d75654a61
2,IT/SPO.IT0477A_6001_BETA_2022-01-01_00:00:00,6001,2025-01-01 02:00:00,2025-01-01 03:00:00,438.117740000000000000,ug.m-3,hour,3,3,2025-01-02 18:00:00,None,ad50fa1d-3abd-4bb0-8360-a37d75654a61
3,IT/SPO.IT0477A_6001_BETA_2022-01-01_00:00:00,6001,2025-01-01 03:00:00,2025-01-01 04:00:00,378.926360000000000000,ug.m-3,hour,3,3,2025-01-02 18:00:00,None,ad50fa1d-3abd-4bb0-8360-a37d75654a61
4,IT/SPO.IT0477A_6001_BETA_2022-01-01_00:00:00,6001,2025-01-01 04:00:00,2025-01-01 05:00:00,305.677300000000000000,ug.m-3,hour,3,3,2025-01-02 18:00:00,None,ad50fa1d-3abd-4bb0-8360-a37d75654a61


In [15]:
# Load downloaded PM2.5 data and summarize its raw quality

PM25_CSV = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "pm25"
    / "PM25_MI_hourly.csv"
)

if not PM25_CSV.exists():
    raise FileNotFoundError(
        f"PM2.5 CSV not found: {PM25_CSV}"
    )

pm25_raw_df = pd.read_csv(PM25_CSV)

pm25_analysis_df = pm25_raw_df.copy()
pm25_analysis_df["Start"] = pd.to_datetime(
    pm25_analysis_df["Start"],
    errors="coerce",
)
pm25_analysis_df["Value"] = pd.to_numeric(
    pm25_analysis_df["Value"],
    errors="coerce",
)

missing_flag_count = (
    pm25_analysis_df["Value"]
    .eq(-9999)
    .sum()
)
missing_flag_percentage = (
    missing_flag_count
    / len(pm25_analysis_df)
    * 100
)

pm25_valid_df = (
    pm25_analysis_df.loc[
        pm25_analysis_df["Start"].notna()
        & pm25_analysis_df["Value"].notna()
        & pm25_analysis_df["Value"].ne(-9999)
    ]
    .sort_values("Start")
    .reset_index(drop=True)
)

valid_rate = (
    len(pm25_valid_df)
    / len(pm25_analysis_df)
    * 100
)

print(f"File: {PM25_CSV.relative_to(PROJECT_ROOT)}")
print(f"Downloaded records: {len(pm25_analysis_df):,}")
print(f"Valid records retained for visualization: {len(pm25_valid_df):,}")
print(f"Valid rate: {valid_rate:.2f}%")
print(f"Missing flags (-9999): {missing_flag_count:,}")
print(f"Missing percentage: {missing_flag_percentage:.2f}%")
print(
    "Temporal coverage:",
    pm25_valid_df["Start"].min(),
    "to",
    pm25_valid_df["Start"].max(),
)

File: data/raw/pm25/PM25_MI_hourly.csv
Downloaded records: 14,160
Valid records retained for visualization: 13,930
Valid rate: 98.38%
Missing flags (-9999): 230
Missing percentage: 1.62%
Temporal coverage: 2025-01-01 00:00:00 to 2026-08-15 23:00:00


## 5. Download and Merge ERA5 data
Download ERA5 meteorological variables and merge single-level and pressure-level data.


In [27]:
import src.data.era5 as era5

importlib.reload(era5)

ERA5_RAW_DIR = PROJECT_ROOT / "data" / "raw" / "era5"

ERA5_CSV = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "era5_all_merged.csv"
)

# Run ERA5 download + processing. 
# monthly chunk outputs and only downloads missing requested date ranges.
START_DATE = "2025-01-01"
END_DATE = "2026-08-16"

if ERA5_CSV.exists():
    print(
        "ERA5 merged file exists. Refreshing requested date range while reusing cached monthly chunks where possible."
    )
else:
    print("ERA5 merged file not found. Starting ERA5 download and processing.")

era5_df = era5.download_era5_data(
    lat=45.4642,
    lon=9.1900,
    start_date=START_DATE,
    end_date=END_DATE,
    work_dir=ERA5_RAW_DIR,
    output_file=ERA5_CSV,
    overwrite=False,
)

print("ERA5 raw files saved to:", ERA5_RAW_DIR.relative_to(PROJECT_ROOT))
print("Merged ERA5 CSV saved to:", ERA5_CSV.relative_to(PROJECT_ROOT))

print(f"Rows: {len(era5_df)}")
print(f"Columns: {len(era5_df.columns)}")

ERA5 merged file exists. Refreshing requested date range while reusing cached monthly chunks where possible.
Using existing ERA5 chunk: era5_merged_20250101_20250131.csv
Using existing ERA5 chunk: era5_merged_20250201_20250228.csv
Using existing ERA5 chunk: era5_merged_20250301_20250331.csv
Using existing ERA5 chunk: era5_merged_20250401_20250430.csv
Using existing ERA5 chunk: era5_merged_20250501_20250531.csv
Using existing ERA5 chunk: era5_merged_20250601_20250630.csv
Using existing ERA5 chunk: era5_merged_20250701_20250731.csv
Using existing ERA5 chunk: era5_merged_20250801_20250831.csv
Using existing ERA5 chunk: era5_merged_20250901_20250930.csv
Using existing ERA5 chunk: era5_merged_20251001_20251031.csv
Using existing ERA5 chunk: era5_merged_20251101_20251130.csv
Using existing ERA5 chunk: era5_merged_20251201_20251231.csv
Using existing ERA5 chunk: era5_merged_20260101_20260131.csv
Using existing ERA5 chunk: era5_merged_20260201_20260228.csv
Using existing ERA5 chunk: era5_merge

2026-08-19 16:57:24,762 INFO Request ID is 8aba1223-ee08-45fe-ba90-ff9d5c57c793
2026-08-19 16:57:24,888 INFO status has been updated to accepted
2026-08-19 16:57:39,812 INFO status has been updated to running
2026-08-19 16:57:47,629 INFO status has been updated to successful


6a64d1ebcccf9d1ee79fcd47723d5e8f.zip:   0%|          | 0.00/12.7k [00:00<?, ?B/s]

2026-08-19 16:57:48,698 INFO Request ID is 317454d7-91e0-42cb-ad46-13b27bed4a14
2026-08-19 16:57:49,345 INFO status has been updated to accepted
2026-08-19 16:58:04,503 INFO status has been updated to running
2026-08-19 17:02:11,561 INFO status has been updated to successful


f3039c207434ed58e2e97143286ed025.zip:   0%|          | 0.00/106k [00:00<?, ?B/s]

Completed ERA5 chunk: 20260801_20260816, 288 rows

ERA5 final quality check
------------------------
Expected rows: 14232
Actual rows: 14136
Missing timestamps: 96
Unexpected timestamps: 0
First missing timestamps:
[Timestamp('2026-08-13 00:00:00'), Timestamp('2026-08-13 01:00:00'), Timestamp('2026-08-13 02:00:00'), Timestamp('2026-08-13 03:00:00'), Timestamp('2026-08-13 04:00:00'), Timestamp('2026-08-13 05:00:00'), Timestamp('2026-08-13 06:00:00'), Timestamp('2026-08-13 07:00:00'), Timestamp('2026-08-13 08:00:00'), Timestamp('2026-08-13 09:00:00')]

Final ERA5 dataset saved
Path: /Users/qingxuan/Desktop/webcam-pm25-benchmark/data/interim/era5_all_merged.csv
Shape: (14136, 20)
Time range: 2025-01-01 00:00:00 to 2026-08-12 23:00:00
ERA5 raw files saved to: data/raw/era5
Merged ERA5 CSV saved to: data/interim/era5_all_merged.csv
Rows: 14136
Columns: 20


## 6. Merge ARPA station tables
ARPA meteorological station data were manually requested from ARPA Lombardia and provided as CSV tables.

This step merges the downloaded station tables into a unified hourly dataset.

In [28]:
import src.data.arpa as arpa

importlib.reload(arpa)

ARPA_ROOT = PROJECT_ROOT / "data" / "raw" / "arpa"

ARPA_FILES = sorted(ARPA_ROOT.rglob("*.csv"))

ARPA_OUTPUT_FILE = PROJECT_ROOT / "data" / "interim" / "arpa_merged.csv"

print(f"Found {len(ARPA_FILES)} ARPA CSV files under: {ARPA_ROOT.relative_to(PROJECT_ROOT)}")

arpa_df = arpa.merge_arpa_tables(
    files=ARPA_FILES,
    output_file=ARPA_OUTPUT_FILE,
    time_column="Data-Ora",
    sensor_column="Id Sensore",
    utc_offset_hours=1,
    missing_value=-999,
)

print("ARPA merged CSV saved to:")
print(ARPA_OUTPUT_FILE.relative_to(PROJECT_ROOT))
print(f"\nRows: {len(arpa_df)}")
print(f"Columns: {arpa_df.shape[1]}")

Found 10 ARPA CSV files under: data/raw/arpa
ARPA merged CSV saved to:
data/interim/arpa_merged.csv

Rows: 14186
Columns: 6


## 7. Merge all datasets
This merges `image_features.csv`, `PM25_MI_hourly.csv`, `era5_all_merged.csv`, and `arpa_merged.csv` by UTC time.

Before the final merge, we confirm that the intermediate source tables have already been normalized to UTC-style timestamps.


In [31]:
# UTC consistency check for all intermediate source tables

check_specs = {
    "image_features": (PROJECT_ROOT / "data" / "interim" / "image_features.csv", "datetime"),
    "pm25": (PM25_CSV, "Start"),
    "arpa": (PROJECT_ROOT / "data" / "interim" / "arpa_merged.csv", "Data-Ora"),
    "era5": (PROJECT_ROOT / "data" / "interim" / "era5_all_merged.csv", "time"),
}

summary = []
for name, (csv_path, time_col) in check_specs.items():
    df = pd.read_csv(csv_path)
    series = pd.to_datetime(df[time_col], errors="coerce")
    summary.append(
        f"{name}: min={series.min()} | max={series.max()} | nulls={series.isna().sum()}"
    )

print("UTC normalization check:")
for line in summary:
    print(line)
print("All source timestamps are normalized to UTC-style naive timestamps before the final merge.")

UTC normalization check:
image_features: min=2025-01-29 15:00:00 | max=2026-08-15 21:00:00 | nulls=0
pm25: min=2025-01-01 00:00:00 | max=2026-08-15 23:00:00 | nulls=0
arpa: min=2024-12-31 23:00:00+00:00 | max=2026-08-15 23:00:00+00:00 | nulls=0
era5: min=2025-01-01 00:00:00 | max=2026-08-12 23:00:00 | nulls=0
All source timestamps are normalized to UTC-style naive timestamps before the final merge.


In [32]:
import src.data.integration as merge_all

importlib.reload(merge_all)

MERGED_OUTPUT = PROJECT_ROOT / "data" / "interim" / "merged_dataset.csv"

merged_df = merge_all.merge_all_datasets(
    arpa_file=PROJECT_ROOT / "data" / "interim" / "arpa_merged.csv",
    image_file=PROJECT_ROOT / "data" / "interim" / "image_features.csv",
    pm25_file=PM25_CSV,
    era5_file=PROJECT_ROOT / "data" / "interim" / "era5_all_merged.csv",
    output_file=MERGED_OUTPUT,
)

print("Merged dataset saved to:")
print(MERGED_OUTPUT.relative_to(PROJECT_ROOT))

print(f"\nRows: {len(merged_df)}")
print(f"Columns: {merged_df.shape[1]}")
print("Merge strategy: image_features is the anchor table; ERA5, PM25, and ARPA are left-merged onto image timestamps.")
print("This keeps the image sample timeline intact and aligns auxiliary variables to those timestamps.")
print("Preview of merged dataset columns and first rows:")
display(merged_df.head(2))

Merged dataset saved to:
data/interim/merged_dataset.csv

Rows: 3862
Columns: 46
Merge strategy: image_features is the anchor table; ERA5, PM25, and ARPA are left-merged onto image timestamps.
This keeps the image sample timeline intact and aligns auxiliary variables to those timestamps.
Preview of merged dataset columns and first rows:


,time,R_roi,G_roi,B_roi,R_std,G_std,B_std,S_mean,V_mean,colorfulness,...,T_850,U_500,U_850,V_500,V_850,temperature_mean,wind_direction_mean,relative_humidity_mean,wind_speed_mean,wind_gust_max
0,2025-01-29 15:00:00,72.890291,102.643112,127.90583,33.593960,41.347283,45.609732,0.445059,0.501627,36.742349,...,274.20013,10.128586,0.558060,-10.743362,4.637741,11.5,100.0,67.6,1.2,3.4
1,2025-01-29 16:00:00,77.456367,108.505774,131.58280,37.263657,48.407776,57.147530,0.409142,0.517590,48.521689,...,274.14580,9.682327,0.860855,-9.707871,5.492218,10.8,91.0,70.2,2.0,3.9
